# Notebook 13: Human genetics: are the interface genes enriched for Alzheimer's risk?

**Goal.** Use gene-level Alzheimer's GWAS results to test the interface genes against random sets of microglia-expressed genes, and test colocalisation with a microglia eQTL (H5).

Dry run: a simulated gene-level table in the same shape as MAGMA output. Real data: run MAGMA first (instructions after the cells) and point the path to *ad_gene.genes.out* joined with gene symbols.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Takes the interface genes from notebook 09 (FDR below 0.1), converts them to human symbols, and loads the gene-level Alzheimer's results (one Z-score per gene; larger = stronger association). **expressed** = genes expressed in human microglia: the comparison pool must be microglia-expressed genes, otherwise any microglia gene set looks 'enriched' just because microglia genes are enriched in Alzheimer's risk.

*You should see:* 21 interface genes mapped to 21 human symbols.

In [ ]:
from mlc.genetics import load_orthologs, to_human, set_enrichment, coloc_abf
from mlc import plots
SIM = RAW / "sim"
orth = load_orthologs(SIM / "HOM_MouseHumanSequence_sim.rpt")
gt = pd.read_csv(SIM / "ad_gene_level_sim.csv")             # real: ad_gene.genes.out merged with symbols (see below)
expressed = pd.read_csv(SIM / "human_microglia_expressed_sim.csv").gene
rew = pd.read_csv(RES / "rewiring_genes.csv")
iface = rew[rew.fdr < 0.1].gene.tolist()
if len(iface) < 5: iface = rew.head(10).gene.tolist(); print("few genes at FDR<0.1; using top 10 by p-value")
hset = to_human(iface, orth); print(len(iface), "interface genes ->", len(hset), "human genes:", hset[:8])

**Cell 3.** Compares the average gene-level Z of the interface genes with 10,000 random sets of the same size drawn from microglia-expressed genes (**empirical p**). The APOE and MHC regions are excluded because they dominate Alzheimer's signal. The same test on an **unrelated trait** (height) is the specificity control: it should be *not* significant.

*You should see:* Alzheimer's: mean Z about 1.6 against a null of about 0.07, p = 0.0001; control trait p about 0.65.

In [ ]:
excl = ((gt.CHR == 19) & gt.START.between(44e6, 46.5e6)) | ((gt.CHR == 6) & gt.START.between(25e6, 34e6))   # APOE, MHC
stat, null = set_enrichment(gt, hset, expressed, n_perm=10000, seed=SEED, exclude=excl)
print("Alzheimer's:", {k: round(v, 4) for k, v in stat.items()})
ctrl = pd.read_csv(SIM / "height_gene_level_sim.csv")       # an unrelated trait as a specificity control
stat_c, null_c = set_enrichment(ctrl, hset, expressed, n_perm=10000, seed=SEED, exclude=excl)
print("control trait:", {k: round(v, 4) for k, v in stat_c.items()})

**Cell 4.** **Colocalisation** asks: do the disease association and the microglia eQTL at a locus come from the *same* causal variant (PP.H4 near 1) or from two nearby different ones (PP.H3 near 1)? The function is a transparent implementation of the standard Bayesian method (approximate Bayes factors; default priors p1=p2=1e-4, p12=1e-5). Each row of the real analysis needs, for one locus, the same SNPs with beta and standard error from both studies.

*You should see:* Ctsd locus: PP.H4 about 1.0; Pink1 locus: PP.H3 about 1.0.

In [ ]:
rows = []
for locus, gene in [("shared", "Ctsd"), ("distinct", "Pink1")]:      # real: one row per candidate gene and its eQTL file
    d = pd.read_csv(SIM / f"coloc_locus_{locus}.csv")
    r = coloc_abf(d.beta_gwas, d.se_gwas, d.beta_eqtl, d.se_eqtl, "quant", "quant")
    rows.append(dict(locus=gene, gene=gene, **{k: round(v, 3) for k, v in r.items()}))
coloc = pd.DataFrame(rows); coloc.to_csv(RES / "coloc_results.csv", index=False); print(coloc)

**Cell 5.** **Figure 6a-b.** (a) Real mean Z (red) against the random expressed-gene null. (b) Posterior probabilities for the two loci.

In [ ]:
plots.fig_genetics(null, stat, coloc); plt.show()